# Week 4: Results table and comparison chart

Reads `fruit_project/results/experiments.csv` (written by `run_experiment` in `03_pipeline.ipynb`) and produces:
- a table with each run's change in macro-F1 compared with the baseline,
- a two-panel bar chart (macro-F1 and miss rate) saved to `results/comparison.png`.

No GPU is needed.

- The grey band on the left chart is the spread of the baseline across different random seeds (the repeats named `baseline_none_frozen_seed1`, `..._seed2`). A difference between two runs that is smaller than that band is not evidence of a real effect.
- Miss rate is the share of diseased fruit called Healthy.
- If a run was repeated by accident or after a crash, list its name in `EXCLUDE` in the first code cell. Otherwise it appears as an extra bar. Only runs named `<baseline>_seed1`, `_seed2`, ... count towards the grey noise band.

In [ ]:
# Load the log
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = "/content/drive/MyDrive/fruit_project"
BASE = "baseline_none_frozen"       # the baseline of record; its seed repeats must be named BASE + "_seed1", "_seed2", ...
EXCLUDE = []                        # runs to leave out entirely
ZOOM = False                        # True: zoom the macro-F1 axis in.

path = f"{PROJECT}/results/experiments.csv"
assert os.path.exists(path), "experiments.csv not found. Run some experiments first."
df = pd.read_csv(path).drop_duplicates("run", keep="last")     # if a run name was logged twice, keep the latest
df = df[~df["run"].isin(EXCLUDE)]
print(len(df), "runs logged")

In [ ]:
# Table: change compared with the baseline, and whether it exceeds the seed-to-seed noise
repeats = df[(df["run"] == BASE) | df["run"].str.startswith(BASE + "_seed")]     # the baseline and its seed repeats only
noise = None
if len(repeats) >= 2:
    noise = repeats["val_macro_f1"].max() - repeats["val_macro_f1"].min()
    print(f"Baseline macro-F1 over {len(repeats)} seeds: {repeats['val_macro_f1'].min():.3f} to "
          f"{repeats['val_macro_f1'].max():.3f} (spread {noise:.3f})")
else:
    print("Only one baseline run logged, so the noise level is unknown. Run the seed repeats from the Week 4 plan.")

main = df[~df["run"].str.contains("_seed")].reset_index(drop=True)     # one row per setting; seed repeats excluded
base_rows = main[main["run"] == BASE]
assert len(base_rows) == 1, f"Could not find a single '{BASE}' row"
base_f1 = base_rows["val_macro_f1"].iloc[0]

main["delta_f1"] = main["val_macro_f1"] - base_f1
if noise is not None:
    main["beyond_noise"] = main["delta_f1"].abs() > noise        # simple rule of thumb, not a statistical test

cols = ["run", "augmentation", "finetune_mode", "val_acc", "val_macro_f1", "delta_f1", "beyond_noise",
        "miss_rate", "false_alarm_rate", "weakest_class", "best_epoch", "sec_per_epoch"]
table = main[[c for c in cols if c in main.columns]].round(3)
display(table)
table.to_csv(f"{PROJECT}/results/comparison_table.csv", index=False)

In [ ]:
# Chart
fig, axes = plt.subplots(1, 2, figsize=(max(10, 1.5 * len(main) + 5), 5.5))
x = list(range(len(main)))

panels = [("val_macro_f1", "Macro-F1 (higher is better)", True),
          ("miss_rate", "Miss rate: diseased called Healthy (lower is better)", False)]
for ax, (col, title, higher_is_better) in zip(axes, panels):
    bars = ax.bar(x, main[col], color="#4C72B0")
    best = main[col].idxmax() if higher_is_better else main[col].idxmin()
    bars[best].set_edgecolor("black"); bars[best].set_linewidth(2.5)          # outline the best run
    for b, v in zip(bars, main[col]):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(main["run"], rotation=45, ha="right")
    ax.set_title(title, fontsize=11)
    if col == "val_macro_f1":
        if ZOOM:
            ax.set_ylim(max(0, main[col].min() - 0.05), min(1.0, main[col].max() + 0.03))
        else:
            ax.set_ylim(0, 1)            # starts at 0 on purpose gaps smaller than the noise should look small
        if noise is not None:
            ax.axhspan(repeats["val_macro_f1"].min(), repeats["val_macro_f1"].max(),
                       color="grey", alpha=0.35, label="baseline seed spread")
            ax.legend(loc="lower right", fontsize=8)
    else:
        ax.set_ylim(0, max(0.05, main[col].max() * 1.3))

plt.tight_layout()
plt.savefig(f"{PROJECT}/results/comparison.png", dpi=150)
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
for run in ["baseline_none_frozen", "ft_full_none"]:
    plt.figure(figsize=(10, 4)); plt.imshow(Image.open(f"{PROJECT}/results/{run}_curves.png"))
    plt.axis("off"); plt.title(run); plt.show()


Model selection:
1. Choice made using validation results only.
2. Compare by macro-F1, not accuracy, because half the photos are Healthy.
3. A difference smaller than the noise is a tie. The noise is the spread of the baseline across three random seeds, which is 0.009 in macro-F1.
4. A tie goes to the simpler model. Simplest to most complex is frozen, then last block, then full fine-tuning, with no augmentation preferred over augmentation.
(FNR rate is not a tie-break because the gap between runs was only about two photos.)

FINAL MODEL:
The frozen ResNet18 baseline (only the last layer is trained, no augmentation, trained for 10 epochs) was chosen.

Reasoning: We compared three ways of training: frozen, last block unfrozen, and all layers unfrozen. Their scores were almost the same (macro-F1 of 0.939, 0.944 and 0.944), and the gaps were smaller than the natural run-to-run change of about 0.009. The share of diseased fruit called healthy was also almost the same (4.5%, 4.5% and 3.7%, which is about two photos apart). Due to a tie, the simplest model was chose.

The fully fine-tuned model fit the training photos almost perfectly within 3 epochs, then stopped improving on validation. It did have a lower validation loss, but that may partly come from validation photos that nearly duplicate training photos.